# Bahdanau Attention — Neural MT by Jointly Learning to Align and Translate (2014)

_Papers · Sequence & NLP_

**Let the decoder look back at every source word and softly pick which ones matter — no single fixed summary vector.**

---

This notebook builds additive attention one step at a time: first the score → softmax → context recipe on a hand worked example, then a full encoder-decoder that learns *where* to look, and finally an ablation that rips the attention out so you can see the fixed-vector bottleneck the paper attacks. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Compute one attention context by hand

Attention is three small operations. First an **alignment score** `e_j` measures how well each source annotation `h_j` matches the current decoder state (eq 7: `v^T tanh(W s + U h_j)`). A **softmax** turns those scores into weights `alpha` that sum to 1 (eq 6). Finally the **context** `c` is the `alpha`-weighted sum of the annotations (eq 5). We run it on three tiny 2-D annotations so you can check every number against the lesson.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Sanity-check the lesson's worked example: score -> softmax -> context.
h = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])    # 3 annotations h_j (2-dim)
s = torch.tensor([0.5, -0.5])                        # decoder state s_{i-1}

Wa = torch.tensor([[0.5, 0.], [0., 0.5]])
Ua = torch.eye(2)
va = torch.tensor([1., 1.])

e = (va * torch.tanh(Wa @ s + h @ Ua.T)).sum(1)      # Eqn. 7  e_j = v^T tanh(W s + U h_j)
alpha = torch.softmax(e, dim=0)                       # Eqn. 6
c = (alpha.unsqueeze(-1) * h).sum(0)                  # Eqn. 5  weighted sum

print("e     =", [round(x, 4) for x in e.tolist()])      # [0.6034, 0.8801, 1.4834]
print("alpha =", [round(x, 4) for x in alpha.tolist()])  # [0.2114, 0.2788, 0.5098]
print("context c =", [round(x, 4) for x in c.tolist()])  # [0.7212, 0.7886]

### Step 2 — Wrap that recipe in an attention module

Now package the same three operations (eqs 5-7) into a reusable `AdditiveAttention` layer. `W` projects the decoder state, `U` projects each source annotation, and `v` collapses the sum to a single score per source position. The softmax runs over the source positions, so every decoder step gets its own distribution over where to look.

In [ ]:
# The additive attention block (built by hand). Eqns. 5-7.
V, L, EMB, HID = 6, 5, 16, 32                         # vocab, seq len, embed, hidden

class AdditiveAttention(nn.Module):
    def __init__(self, hid, ann):                     # ann = annotation dim (= 2*hid, bidirectional)
        super().__init__()
        self.W = nn.Linear(hid, hid, bias=False)      # W_a s_{i-1}
        self.U = nn.Linear(ann, hid, bias=False)      # U_a h_j
        self.v = nn.Linear(hid, 1, bias=False)        # v_a^T

    def forward(self, s, H):                           # s:(N,hid)  H:(N,L,ann)
        scores = self.v(torch.tanh(self.W(s).unsqueeze(1) + self.U(H)))  # Eqn.7
        e = scores.squeeze(-1)                         # -> (N, L)
        alpha = torch.softmax(e, dim=1)               # Eqn. 6  (normalize over source positions)
        c = (alpha.unsqueeze(-1) * H).sum(1)          # Eqn. 5  weighted sum -> (N, ann)
        return c, alpha

### Step 3 — Build the encoder and the attentional decoder

A bidirectional GRU encoder turns the source sequence into annotations `h_j`. The decoder steps through the target one token at a time; at each step it asks the attention module for a **fresh context** and feeds `[embedding, context]` into a GRU cell. The `attend=False` branch is the ablation: it ignores attention and reuses one fixed encoder vector for every step — the plain seq2seq bottleneck.

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, EMB)
        self.rnn = nn.GRU(EMB, HID, batch_first=True, bidirectional=True)

    def forward(self, x):
        annotations, _ = self.rnn(self.emb(x))
        return annotations                            # h_j -> (N, L, 2*HID)


class Decoder(nn.Module):
    def __init__(self, attend=True):
        super().__init__()
        self.attend = attend
        self.emb  = nn.Embedding(V, EMB)
        self.attn = AdditiveAttention(HID, 2 * HID)
        self.cell = nn.GRUCell(EMB + 2 * HID, HID)
        self.out  = nn.Linear(HID, V)

    def forward(self, H, tgt):                         # teacher forcing on tgt
        N = H.size(0)
        s   = torch.zeros(N, HID)
        inp = torch.zeros(N, dtype=torch.long)         # BOS = 0
        logits, attns = [], []
        for t in range(L):
            if self.attend:
                c, a = self.attn(s, H)                  # fresh context EVERY step
            else:
                c = H[:, -1, :]                          # ABLATION: one fixed vector (plain seq2seq)
                a = torch.zeros(N, L)
            s = self.cell(torch.cat([self.emb(inp), c], -1), s)
            logits.append(self.out(s))
            attns.append(a)
            inp = tgt[:, t]
        return torch.stack(logits, 1), torch.stack(attns, 1)   # (N,L,V), (N,L,L)

### Step 4 — Train on a copy task and run the ablation

We train on a toy **copy** task (output = input) where nothing tells the model the alignment — it has to discover that step `i` should attend to source position `i`. After training we read off copy accuracy and the average alignment matrix, which should be diagonal-dominant. Then we retrain with `attend=False`: stripping attention down to one fixed vector hurts accuracy, isolating attention as the cause.

In [ ]:
# Toy COPY task: output = input. Nothing tells the model the alignment.
def make(n):
    return torch.randint(1, V, (n, L))

def train(attend, epochs=15, N=3000):
    torch.manual_seed(0)
    enc, dec = Encoder(), Decoder(attend=attend)
    opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=3e-3)
    lf  = nn.CrossEntropyLoss()
    data = make(N)
    for ep in range(epochs):
        perm = torch.randperm(N)
        for i in range(0, N, 128):
            b = data[perm[i:i+128]]
            logits, _ = dec(enc(b), b)                  # copy: target == source
            loss = lf(logits.reshape(-1, V), b.reshape(-1))
            opt.zero_grad()
            loss.backward()
            opt.step()
    # evaluate copy accuracy + average alignment matrix
    test = make(500)
    with torch.no_grad():
        logits, attns = dec(enc(test), test)
        acc = (logits.argmax(-1) == test).float().mean().item()
        A = attns.mean(0)                               # avg over test set -> (L_dec, L_enc)
    return enc, dec, acc, A

enc, dec, acc, A = train(attend=True)
print("ATTENTION copy token accuracy:", round(acc, 4))
print("avg alignment matrix (rows = decoder step, cols = source pos):")
for row in A.tolist():
    print("  ", [round(x, 3) for x in row])

# ABLATION: single fixed context vector (the fixed-length bottleneck the paper attacks).
_, _, acc0, _ = train(attend=False)
print("\nFIXED-CONTEXT (ablation) copy token accuracy:", round(acc0, 4))
print("Attention is near-perfect with a near-diagonal heatmap; the fixed-context model copies worse.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_On a copy task (output = input), what shape does the learned alignment matrix α take — and does it confirm the model learns WHERE to look?_

### Step 1 — Redefine the model for a self-contained run

This visualization cell stands alone, so it re-imports torch and redefines the encoder, the additive-attention block, and the decoder (attention-only this time). Re-seeding keeps it reproducible. The logic is identical to the reference cell — it is regrouped here so the alignment heatmap can be produced on its own.

In [ ]:
import torch
import torch.nn as nn

# Reproduces the qualitative effect: a learned, near-diagonal alignment on a copy task.
torch.manual_seed(0)
V, L, EMB, HID, N = 6, 5, 16, 32, 3000

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, EMB)
        self.rnn = nn.GRU(EMB, HID, batch_first=True, bidirectional=True)

    def forward(self, x):
        return self.rnn(self.emb(x))[0]

class AdditiveAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W = nn.Linear(HID, HID, bias=False)
        self.U = nn.Linear(2 * HID, HID, bias=False)
        self.v = nn.Linear(HID, 1, bias=False)

    def forward(self, s, H):
        e = self.v(torch.tanh(self.W(s).unsqueeze(1) + self.U(H))).squeeze(-1)  # Eqn.7
        a = torch.softmax(e, dim=1)                                             # Eqn.6
        return (a.unsqueeze(-1) * H).sum(1), a                                  # Eqn.5

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(V, EMB)
        self.attn = AdditiveAttention()
        self.cell = nn.GRUCell(EMB + 2 * HID, HID)
        self.out  = nn.Linear(HID, V)

    def forward(self, H, tgt):
        n = H.size(0)
        s = torch.zeros(n, HID)
        inp = torch.zeros(n, dtype=torch.long)
        lo, at = [], []
        for t in range(L):
            c, a = self.attn(s, H)
            s = self.cell(torch.cat([self.emb(inp), c], -1), s)
            lo.append(self.out(s))
            at.append(a)
            inp = tgt[:, t]
        return torch.stack(lo, 1), torch.stack(at, 1)

### Step 2 — Train the copy model

Build the encoder and decoder, then train on the same toy copy task for 15 epochs with Adam and cross-entropy loss. Every detail here matches the reference run; this is just the training loop on its own so the next step can read off the result.

In [ ]:
enc, dec = Encoder(), Decoder()
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=3e-3)
lf  = nn.CrossEntropyLoss()
data = torch.randint(1, V, (N, L))

for ep in range(15):
    perm = torch.randperm(N)
    for i in range(0, N, 128):
        b = data[perm[i:i+128]]
        logits, _ = dec(enc(b), b)
        loss = lf(logits.reshape(-1, V), b.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()

### Step 3 — Read the alignment matrix

Evaluate on fresh sequences, print the copy accuracy, and average the per-example attention into one alignment matrix. It should be **diagonal-dominant**: decoder step `i` puts most of its weight on source position `i`, the visual confirmation that the model learned *where* to look.

In [ ]:
test = torch.randint(1, V, (500, L))
with torch.no_grad():
    logits, attns = dec(enc(test), test)
    acc = (logits.argmax(-1) == test).float().mean().item()
    A = attns.mean(0)                       # avg alignment matrix

print("copy accuracy:", round(acc, 4))      # ~0.999
for row in A.tolist():
    print([round(x, 3) for x in row])
# Diagonal-dominant: step i attends to source position i. Our run, not the paper's numbers.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working attention copy model whose alignment heatmap is bright on
            the diagonal. Replace the per-step attention with a single fixed context (use the last
            encoder annotation for every decoder step, plain-seq2seq style) and retrain. What happens to copy
            accuracy and to the heatmap, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap the attention call for a fixed vector: set c_i = H[:, -1, :] (or a learned constant) for every output step, deleting the score/softmax/weighted-sum. — _An honest ablation changes exactly one thing &mdash; the attention &mdash; so any difference is attributable to it. This recreates the fixed-length bottleneck the paper attacks._
- Retrain with everything else identical and measure per-token copy accuracy; also try to plot an alignment matrix. — _With no per-step weights there is no alignment to plot, and one vector must carry every source position at once._
- Compare: the attention model copies near-perfectly with a near-diagonal heatmap; the fixed-context model copies worse (especially later positions) and has no usable alignment. — _The single vector cannot hold all positions distinctly, so the decoder loses track of which word to emit where &mdash; the bottleneck reproduced on toy data._

**Answer:** Copy accuracy drops (later positions suffer most) and the alignment structure
                 disappears &mdash; there are no per-step weights to form a diagonal. Since the two models
                 are identical except for "per-step attention vs one fixed context," this isolates attention
                 as the reason the decoder can reliably pick the right source word at each step. It reproduces
                 the paper's core claim: the single fixed vector is a bottleneck.

</details>

**Problem 2.** Your worked example had $\alpha = [0.211, 0.279, 0.510]$ and annotations $h_1=[1,0]$,
            $h_2=[0,1]$, $h_3=[1,1]$, giving context $c=[0.721,0.789]$. Suppose the alignment network instead
            produced scores so extreme that softmax returned $\alpha = [0,0,1]$. What is the context now, and
            what does that limiting case tell you about attention?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug into Eqn. 5: $c = 0\cdot[1,0] + 0\cdot[0,1] + 1\cdot[1,1] = [1,1]$. — _A one-hot $\alpha$ makes the weighted sum equal to a single annotation &mdash; here $h_3$._
- Note this equals "hard-select source word 3" &mdash; the discrete alignment a classical aligner would make. — _Softmax attention contains hard alignment as its sharp limit; usually it stays soft (spread mass) so gradients can flow._
- Contrast with the actual $\alpha=[0.211,0.279,0.510]$: the real context $[0.721,0.789]$ blends all three, mostly $h_3$. — _Soft weights let the model hedge and stay differentiable; the hard limit is the special case of total confidence._

**Answer:** With $\alpha=[0,0,1]$ the context is $c=[1,1]=h_3$ &mdash; attention collapses to a
                 hard pick of source word 3. So softmax attention is a soft, differentiable
                 generalization of hard alignment: its sharp limit is "choose exactly one word," but staying
                 soft keeps gradients alive so the alignment network can be trained.

</details>

**Problem 3.** In the alignment score $e_{ij} = v_a^{\top}\tanh(W_a s_{i-1} + U_a h_j)$, the term
            $W_a s_{i-1}$ does not depend on $j$. A classmate says "then it cannot affect which source word
            wins, so we can drop it." Are they right?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note $W_a s_{i-1}$ is a constant vector added inside the $\tanh$ for all $j$, but $\tanh$ is nonlinear. — _If the score were linear in its argument, a $j$-independent additive constant would shift all scores equally and softmax would cancel it. Nonlinearity breaks that._
- Observe that $\tanh(W_a s + U_a h_j)$ bends differently depending on where $W_a s$ shifts the input &mdash; so the same $h_j$ can score high for one decoder state and low for another. — _The decoder state is exactly what tells the model "where I am in the translation" and hence where to look; it must enter the score._
- Conclude that dropping $W_a s_{i-1}$ would make every output step share the same alignment &mdash; the heatmap rows would be identical. — _Without the query term the attention can no longer move along the source as the decoder advances._

**Answer:** No. Although $W_a s_{i-1}$ is the same for all $j$ at a given step, it sits inside the
                 nonlinear $\tanh$, so it changes the relative scores across $j$ &mdash; a constant
                 inside $\tanh$ does not cancel the way a constant outside softmax would. It is precisely the
                 decoder state that lets attention shift to a different source word at each output step; remove
                 it and every row of the alignment matrix becomes identical.

</details>